In [2]:
# ============================================================
# CELL 1: SETUP, TRAINING & VALIDATION EVALUATION (LINEAR SVM)
# ============================================================

import os
import json
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# 1. CONFIGURATION
RANDOM_STATE = 42
ARTIFACT_DIR = "./ML_Artifacts"
OUTPUT_DIR = "./SVM_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. LOAD DATA
print("=" * 70)
print("LOADING DATA ARTIFACTS")
print("=" * 70)

X_train = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_train_tfidf.npz"))
X_val   = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_val_tfidf.npz"))
X_test  = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_test_tfidf.npz"))

y_train = np.load(os.path.join(ARTIFACT_DIR, "y_train.npy"))
y_val   = np.load(os.path.join(ARTIFACT_DIR, "y_val.npy"))
y_test  = np.load(os.path.join(ARTIFACT_DIR, "y_test.npy"))

mapping_path = os.path.join(ARTIFACT_DIR, "class_mapping.json")
if os.path.exists(mapping_path):
    with open(mapping_path, "r", encoding="utf-8") as f:
        mapping_data = json.load(f)
    id_to_class = {int(k): v for k, v in mapping_data["id_to_class"].items()}
    class_names = [id_to_class[i] for i in range(len(id_to_class))]
else:
    class_names = ["SQLInjection", "XSS", "CommandInjection", "Normal"]

print(f"X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   | y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}  | y_test  : {y_test.shape}")


# 3. REUSABLE EVALUATION FUNCTION
def evaluate_dataset(model, X, y, dataset_name):
    print("\n" + "=" * 70)
    print(f"Linear SVM - {dataset_name}")
    print("=" * 70)

    start_time = time.time()
    y_pred = model.predict(X)
    pred_time = time.time() - start_time

    acc = accuracy_score(y, y_pred)
    prec_macro = precision_score(y, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy       : {acc:.4f}")
    print(f"Precision Macro: {prec_macro:.4f}")
    print(f"Recall Macro   : {rec_macro:.4f}")
    print(f"F1 Macro       : {f1_macro:.4f}")
    print(f"F1 Weighted    : {f1_weighted:.4f}")
    print(f"Prediction Time: {pred_time:.4f} seconds")

    # Classification Report
    report_dict = classification_report(
        y, y_pred, target_names=class_names, digits=6, zero_division=0, output_dict=True
    )
    report_df = pd.DataFrame(report_dict).transpose()
    print("\nClassification Report:")
    display(report_df)

    # Confusion Matrix
    cm = confusion_matrix(y, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Save Artifacts
    report_path = os.path.join(OUTPUT_DIR, f"SVM_{dataset_name}_ClassificationReport.csv")
    report_df.to_csv(report_path)

    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_path = os.path.join(OUTPUT_DIR, f"SVM_{dataset_name}_ConfusionMatrix.csv")
    cm_df.to_csv(cm_path, encoding="utf-8-sig")

    cmd_idx = class_names.index("CommandInjection") if "CommandInjection" in class_names else 2
    cmd_recall = recall_score(y, y_pred, labels=[cmd_idx], average="macro", zero_division=0)
    cmd_f1 = f1_score(y, y_pred, labels=[cmd_idx], average="macro", zero_division=0)

    summary_metrics = {
        "Model": "LinearSVC",
        "Dataset": dataset_name,
        "Accuracy": acc,
        "Precision_Macro": prec_macro,
        "Recall_Macro": rec_macro,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted,
        "Command_Recall": cmd_recall,
        "Command_F1": cmd_f1,
        "Prediction_Time": pred_time
    }
    return summary_metrics


# 4. TRAIN LINEAR SVM
print("\n" + "=" * 70)
print("TRAINING LINEAR SVM")
print("=" * 70)

svm_model = LinearSVC(
    C=1.0,
    loss="squared_hinge",
    max_iter=5000,
    random_state=RANDOM_STATE
)

start_train = time.time()
svm_model.fit(X_train, y_train)
svm_training_time = time.time() - start_train
print(f"Linear SVM Training Time: {svm_training_time:.2f} seconds ({svm_training_time / 60:.2f} minutes)")

# Save Model
with open(os.path.join(OUTPUT_DIR, "SVM_Model.pkl"), "wb") as f:
    pickle.dump(svm_model, f)

# 5. RUN VALIDATION
val_result = evaluate_dataset(svm_model, X_val, y_val, "Validation")
val_result["Training_Time"] = svm_training_time

val_df = pd.DataFrame([val_result])
val_df.to_csv(os.path.join(OUTPUT_DIR, "SVM_Validation_Summary.csv"), index=False)
print(f"\n✓ บันทึกผลลัพธ์ Validation เรียบร้อยใน: {OUTPUT_DIR}")

LOADING DATA ARTIFACTS
X_train : (35000, 30000) | y_train : (35000,)
X_val   : (7500, 30000)   | y_val   : (7500,)
X_test  : (7500, 30000)  | y_test  : (7500,)

TRAINING LINEAR SVM
Linear SVM Training Time: 11.65 seconds (0.19 minutes)

Linear SVM - Validation
Accuracy       : 0.9699
Precision Macro: 0.9718
Recall Macro   : 0.9699
F1 Macro       : 0.9701
F1 Weighted    : 0.9701
Prediction Time: 0.0302 seconds

Classification Report:


,precision,recall,f1-score,support
SQLInjection,0.993482,0.975467,0.984392,1875.000000
XSS,0.998395,0.995200,0.996795,1875.000000
CommandInjection,0.990826,0.921600,0.954960,1875.000000
Normal,0.904692,0.987200,0.944147,1875.000000
accuracy,0.969867,0.969867,0.969867,0.969867
macro avg,0.971849,0.969867,0.970073,7500.000000
weighted avg,0.971849,0.969867,0.970073,7500.000000



Confusion Matrix:
[[1829    0    4   42]
 [   0 1866    2    7]
 [   1    0 1728  146]
 [  11    3   10 1851]]

✓ บันทึกผลลัพธ์ Validation เรียบร้อยใน: ./SVM_Results


In [3]:
# ============================================================
# CELL 2: FINAL TEST EVALUATION (LINEAR SVM)
# ============================================================

print("=" * 70)
print("EVALUATING FINAL TEST SET")
print("=" * 70)

# รันประเมิน Test Set
test_result = evaluate_dataset(svm_model, X_test, y_test, "Test")
test_result["Training_Time"] = svm_training_time

# บันทึกสรุปผล Test Set แยกต่างหาก
test_df = pd.DataFrame([test_result])
test_df.to_csv(os.path.join(OUTPUT_DIR, "SVM_Test_Summary.csv"), index=False)

# แสดงตารางเปรียบเทียบ Val vs Test
comparison_df = pd.DataFrame([val_result, test_result])
print("\n" + "=" * 70)
print("VALIDATION VS TEST COMPARISON")
print("=" * 70)
display(comparison_df[["Dataset", "Accuracy", "F1_Macro", "Command_Recall", "Command_F1", "Prediction_Time"]].round(4))

print(f"\n✓ บันทึกผลลัพธ์ Test Set เรียบร้อยใน: {OUTPUT_DIR}")

EVALUATING FINAL TEST SET

Linear SVM - Test
Accuracy       : 0.9708
Precision Macro: 0.9727
Recall Macro   : 0.9708
F1 Macro       : 0.9710
F1 Weighted    : 0.9710
Prediction Time: 0.0302 seconds

Classification Report:


,precision,recall,f1-score,support
SQLInjection,0.997814,0.973867,0.985695,1875.0000
XSS,0.998930,0.995733,0.997329,1875.0000
CommandInjection,0.988048,0.925867,0.955947,1875.0000
Normal,0.906067,0.987733,0.945139,1875.0000
accuracy,0.970800,0.970800,0.970800,0.9708
macro avg,0.972715,0.970800,0.971028,7500.0000
weighted avg,0.972715,0.970800,0.971028,7500.0000



Confusion Matrix:
[[1826    0    3   46]
 [   0 1867    1    7]
 [   0    0 1736  139]
 [   4    2   17 1852]]

VALIDATION VS TEST COMPARISON


,Dataset,Accuracy,F1_Macro,Command_Recall,Command_F1,Prediction_Time
0,Validation,0.9699,0.9701,0.9216,0.9550,0.0302
1,Test,0.9708,0.9710,0.9259,0.9559,0.0302



✓ บันทึกผลลัพธ์ Test Set เรียบร้อยใน: ./SVM_Results
